In [1]:
!pip install ultralytics

In [2]:
from ultralytics.data.utils import check_det_dataset

# Perintah ini akan menyuruh Ultralytics mencari VisDrone.yaml bawaannya,
# lalu secara otomatis mendownload dan mengekstrak dataset ke folder /content/datasets/VisDrone
print("⏳ Mengunduh dataset VisDrone bawaan Ultralytics...")
dataset_info = check_det_dataset('VisDrone.yaml')
print("✅ Selesai! Dataset raw VisDrone sudah siap.")

In [3]:
import os
import shutil
from tqdm import tqdm
import yaml

# ==========================================================
# 1. KONFIGURASI DIREKTORI DAN MAPPING KELAS
# ==========================================================
SOURCE_VISDRONE_DIR = "/kaggle/working/datasets/VisDrone" 
TARGET_DIR = "/kaggle/working/datasets/VisDrone_1"    

# MAPPING KELAS VISDRONE KE FORMAT BARU
# MAPPING KELAS VISDRONE KE FORMAT KETAT (PERSON & VEHICLE RODA 4+)
CLASS_MAP = {
    # Target Kelas 0: Person (Pedestrian, People, Bicycle)
    0: 0,
    1: 0,
    2: 0,

    # Target Kelas 1: Vehicle (Car, Van, Truck, Bus)
    # Catatan: Kelas 6, 7 (Tricycle) dan 9 (Motor) sengaja TIDAK DIMASUKKAN
    # sehingga akan otomatis difilter/dibuang oleh skrip.
    3: 1,
    4: 1,
    5: 1,
    8: 1
}
# Subfolder bawaan Ultralytics untuk VisDrone
SPLITS = {
    'train': 'VisDrone2019-DET-train',
    'val': 'VisDrone2019-DET-val',
    'test': 'VisDrone2019-DET-test-dev'
}

# ==========================================================
# 2. PERSIAPAN FOLDER DAN PEMROSESAN PER SPLIT
# ==========================================================
for split_name, split_folder in SPLITS.items():
    print(f"\n🚀 Memproses set: {split_name.upper()}...")
    
    # Path Sumber
    src_img_dir = os.path.join(SOURCE_VISDRONE_DIR, split_folder, 'images')
    src_lbl_dir = os.path.join(SOURCE_VISDRONE_DIR, split_folder, 'labels')
    
    # Perbaikan: Ultralytics kadang mengekstrak langsung ke images/train, dsb.
    if not os.path.exists(src_img_dir):
        src_img_dir = os.path.join(SOURCE_VISDRONE_DIR, 'images', split_folder)
        src_lbl_dir = os.path.join(SOURCE_VISDRONE_DIR, 'labels', split_folder)
        # Jika nama foldernya train/val/test (format standar ultralytics baru)
        if not os.path.exists(src_img_dir):
             src_img_dir = os.path.join(SOURCE_VISDRONE_DIR, 'images', split_name)
             src_lbl_dir = os.path.join(SOURCE_VISDRONE_DIR, 'labels', split_name)

    # Path Target
    tgt_img_dir = os.path.join(TARGET_DIR, 'images', split_name)
    tgt_lbl_dir = os.path.join(TARGET_DIR, 'labels', split_name)
    os.makedirs(tgt_img_dir, exist_ok=True)
    os.makedirs(tgt_lbl_dir, exist_ok=True)
    
    if not os.path.exists(src_img_dir):
        print(f"⚠️ Folder sumber untuk {split_name} tidak ditemukan. Dilewati.")
        continue
        
    img_files = [f for f in os.listdir(src_img_dir) if f.endswith(('.jpg', '.png'))]
    count_valid = 0
    
    for img_name in tqdm(img_files, desc=f"Konversi {split_name}"):
        base_name = os.path.splitext(img_name)[0]
        lbl_path = os.path.join(src_lbl_dir, base_name + '.txt')
        
        new_labels = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        old_cls_id = int(parts[0])
                        if old_cls_id in CLASS_MAP:
                            new_cls_id = CLASS_MAP[old_cls_id]
                            new_labels.append(f"{new_cls_id} {' '.join(parts[1:])}\n")
        
        # Hanya salin jika gambar punya label valid
        if new_labels:
            shutil.copy(os.path.join(src_img_dir, img_name), os.path.join(tgt_img_dir, img_name))
            with open(os.path.join(tgt_lbl_dir, base_name + '.txt'), 'w') as f:
                f.writelines(new_labels)
            count_valid += 1
            
    print(f"✅ Set {split_name}: Berhasil memproses {count_valid} gambar valid.")

# ==========================================================
# 3. PEMBUATAN FILE DATA.YAML
# ==========================================================
yaml_content = {
    'path': TARGET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': ['person', 'vehicle']
}

yaml_path = os.path.join(TARGET_DIR, 'visdrone_person_vehicle.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"\n🎉 SELESAI! Konfigurasi YAML tersimpan di: {yaml_path}")


🚀 Memproses set: TRAIN...


Konversi train: 100%|██████████| 6471/6471 [00:02<00:00, 2442.19it/s]


✅ Set train: Berhasil memproses 6471 gambar valid.

🚀 Memproses set: VAL...


Konversi val: 100%|██████████| 548/548 [00:00<00:00, 2369.70it/s]


✅ Set val: Berhasil memproses 548 gambar valid.

🚀 Memproses set: TEST...


Konversi test: 100%|██████████| 1610/1610 [00:00<00:00, 2704.74it/s]

✅ Set test: Berhasil memproses 1610 gambar valid.

🎉 SELESAI! Konfigurasi YAML tersimpan di: /kaggle/working/datasets/VisDrone_1/visdrone_person_vehicle.yaml


In [4]:
!zip -r VisDrone.zip /kaggle/working/datasets/VisDrone_1